In [ ]:
! pip install -U accelerate
! pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 99.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [ ]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 18.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.


In [ ]:
!pip install pyvi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.7 MB/s eta 0:00:00


In [ ]:
!nvidia-smi

Wed May  7 08:56:48 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import wandb
import json
import random
import unicodedata
from pyvi import ViTokenizer
import accelerate
import re
import time
from multiprocessing import Pool
from functools import partial
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import AutoModelWithLMHead,EarlyStoppingCallback, AutoConfig
from sklearn.model_selection import train_test_split
from datasets import load_dataset
from transformers import DataCollatorForLanguageModeling
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
import torch
import pandas as pd
import numpy as np
from tqdm.contrib.concurrent import thread_map

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("NlpHUST/gpt2-vietnamese")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/854k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/512k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

In [ ]:
SPECIAL_TOKENS = ["<|question|>", "<|response|>"]

tokenizer.add_tokens(SPECIAL_TOKENS, special_tokens=True)

tokenizer.add_special_tokens({
    'bos_token': '<|startoftext|>',
    'eos_token': '<|endoftext|>',
    'pad_token': '<PAD>',
    'unk_token': '<UNK>'
})

model_config = AutoConfig.from_pretrained(
    'NlpHUST/gpt2-vietnamese',
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    unk_token_id=tokenizer.pad_token_id,
    output_hidden_states=False
)

config.json:   0%|          | 0.00/884 [00:00<?, ?B/s]

In [ ]:
print(tokenizer.special_tokens_map)

{'bos_token': '<|startoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<UNK>', 'pad_token': '<PAD>', 'additional_special_tokens': ['<|question|>', '<|response|>']}


In [ ]:
print(tokenizer.encode("<|startoftext|>"))
print(tokenizer.encode("<|endoftext|>"))
print(tokenizer.encode("<|question|>"))
print(tokenizer.encode("<|response|>"))
print(tokenizer.encode("<PAD>"))
print(tokenizer.encode("<UNK>"))

[50260]
[50257]
[50258]
[50259]
[50261]
[50262]


In [ ]:
import json
import pandas as pd

input_path = "/content/drive/MyDrive/NLPfinal/data_40k.json"
output_path = "/content/drive/MyDrive/NLPfinal/data_qna_clean.json"

# Load the JSON file
with open(input_path, 'r', encoding='utf-8') as f:
    data = json.load(f)  # Loads a list of strings

# Parse strings into dictionaries
def parse_qa_string(item):
    try:
        # Split on "response:" to separate question and response
        parts = item.split("response:", 1)
        if len(parts) == 2:
            # Extract question (remove "question:" prefix)
            question = parts[0].replace("question:", "", 1).strip()
            response = parts[1].strip()
            return {"question": question, "response": response}
        else:
            # Return empty values instead of "text" for malformed strings
            return {"question": "", "response": ""}
    except:
        # Handle any errors with empty values
        return {"question": "", "response": ""}

# Convert all items to dictionaries
parsed_data = []
for item in data:
    if isinstance(item, str):
        parsed_data.append(parse_qa_string(item))
    elif isinstance(item, dict) and "question" in item and "response" in item:
        parsed_data.append({"question": item["question"], "response": item["response"]})
    else:
        # Skip or assign empty values for unexpected types
        parsed_data.append({"question": "", "response": ""})

# Create DataFrame
df = pd.DataFrame(parsed_data)
print(df.head())  # Inspect the DataFrame
print("Columns:", df.columns.tolist())  # Verify columns

# Save to output file
df.to_json(output_path, orient='records', lines=True, force_ascii=False)

                                            question  \
0  Dạo này mình thấy đầu óc cứ lơ đễnh sao ấy. Ng...   
1  Người mình lúc nào cũng bồn chồn, không ngồi y...   
2  Mình hay làm gì đó theo cảm hứng lắm, kiểu chư...   
3  Mình thấy mình rất tệ trong việc sắp xếp mọi t...   
4  Nhiều lúc mình dễ bị tức giận hoặc buồn bã đột...   

                                            response  
0  Mình hiểu cảm giác bị sao nhãng làm việc không...  
1  Cảm giác bồn chồn, không ngồi yên được nó mệt ...  
2  Việc hành động bốc đồng mà sau đó lại thấy hối...  
3  Vụ tổ chức công việc hay sắp xếp mọi thứ lộn x...  
4  Việc cảm xúc cứ như tàu lượn siêu tốc, lúc lên...  
Columns: ['question', 'response']


In [ ]:

print(df.head())

def process(index, item):
    for key in ['question', 'response']:
        text = item[key]

        # Chuẩn hóa NFC
        text = unicodedata.normalize('NFC', text)

        # Chuyển chữ thường
        text = text.lower()

        # Loại ký tự đặc biệt (giữ lại .,!? và chữ cái)
        text = re.sub(r'[^\w\s,.!?]', '', text)

        # Gom khoảng trắng
        text = re.sub(r'\s+', ' ', text)

        # Loại dấu câu lặp
        text = re.sub(r'([,.!?])\1+', r'\1', text)

        # Tách từ tiếng Việt
        text = ViTokenizer.tokenize(text)

        # Xóa khoảng trắng đầu/cuối
        text = text.strip()

        item[key] = text
    return item


                                            question  \
0  Dạo này mình thấy đầu óc cứ lơ đễnh sao ấy. Ng...   
1  Người mình lúc nào cũng bồn chồn, không ngồi y...   
2  Mình hay làm gì đó theo cảm hứng lắm, kiểu chư...   
3  Mình thấy mình rất tệ trong việc sắp xếp mọi t...   
4  Nhiều lúc mình dễ bị tức giận hoặc buồn bã đột...   

                                            response  
0  Mình hiểu cảm giác bị sao nhãng làm việc không...  
1  Cảm giác bồn chồn, không ngồi yên được nó mệt ...  
2  Việc hành động bốc đồng mà sau đó lại thấy hối...  
3  Vụ tổ chức công việc hay sắp xếp mọi thứ lộn x...  
4  Việc cảm xúc cứ như tàu lượn siêu tốc, lúc lên...  


In [ ]:
results = thread_map(lambda x: process(x[0], x[1]), df.iterrows(), max_workers=20)

0it [00:00, ?it/s]

In [ ]:
pd.DataFrame(results).to_json(output_path, orient='records', force_ascii=False, indent=2)

In [ ]:
def read_json_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

# Chuyển đổi thành định dạng yêu cầu
def format_data(data):
    formatted = []
    for item in data:
        question = item['question']
        response = item['response']
        formatted.append(f"<|startoftext|><|question|> {question} <|response|> {response} <|endoftext|>\n")
    return formatted

# Ghi dữ liệu ra file txt
def write_to_txt(data, output_file):
    with open(output_file, 'w', encoding='utf-8') as f:
        f.writelines(data)

# Xử lý chính
def process_json_to_txt(json_file, train_output, test_output, test_size=0.2):
    # Đọc dữ liệu
    data = read_json_file(json_file)

    # Chuyển đổi định dạng
    formatted_data = format_data(data)

    # Chia train/test
    train_data, test_data = train_test_split(formatted_data, test_size=test_size, random_state=42)

    # Ghi ra file
    write_to_txt(train_data, train_output)
    write_to_txt(test_data, test_output)

    print(f"Đã tạo: {train_output} ({len(train_data)} dòng)")
    print(f"Đã tạo: {test_output} ({len(test_data)} dòng)")

In [ ]:
json_file = '/content/drive/MyDrive/NLPfinal/data_qna_clean.json'
train_output = '/content/drive/MyDrive/NLPfinal/train.txt'
test_output = '/content/drive/MyDrive/NLPfinal/valid.txt'

In [ ]:
process_json_to_txt(json_file, train_output, test_output, test_size=0.2)

Đã tạo: /content/drive/MyDrive/NLPfinal/train.txt (11225 dòng)
Đã tạo: /content/drive/MyDrive/NLPfinal/valid.txt (2807 dòng)


In [ ]:
def analyze_text_file(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    print(f"File: {filename}")
    print(f"Tổng số dòng: {len(lines)}")

    question_lengths = []
    response_lengths = []
    unique_pairs = set()

    for line in lines:
        if '<|question|>' in line and '<|response|>' in line:
            question = line.split('<|question|>')[1].split('<|response|>')[0]
            response = line.split('<|response|>')[1].split('<|endoftext|>')[0]
            question_lengths.append(len(question))
            response_lengths.append(len(response))
            unique_pairs.add((question, response))

    print(f"Độ dài trung bình câu hỏi: {sum(question_lengths) / len(question_lengths):.2f} ký tự")
    print(f"Độ dài trung bình câu trả lời: {sum(response_lengths) / len(response_lengths):.2f} ký tự")
    print(f"Số cặp question-response duy nhất: {len(unique_pairs)}")

# Thống kê
analyze_text_file('/content/drive/MyDrive/NLPfinal/train.txt')
analyze_text_file('/content/drive/MyDrive/NLPfinal/valid.txt')

File: /content/drive/MyDrive/NLPfinal/train.txt
Tổng số dòng: 11225
Độ dài trung bình câu hỏi: 164.31 ký tự
Độ dài trung bình câu trả lời: 587.01 ký tự
Số cặp question-response duy nhất: 11181
File: /content/drive/MyDrive/NLPfinal/valid.txt
Tổng số dòng: 2807
Độ dài trung bình câu hỏi: 161.50 ký tự
Độ dài trung bình câu trả lời: 579.54 ký tự
Số cặp question-response duy nhất: 2803


In [ ]:
model = AutoModelForCausalLM.from_pretrained("NlpHUST/gpt2-vietnamese", config= model_config)

pytorch_model.bin:   0%|          | 0.00/510M [00:00<?, ?B/s]

In [ ]:
model.resize_token_embeddings(len(tokenizer))

print(model.transformer.wte.weight.shape)

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


model.safetensors:   0%|          | 0.00/510M [00:00<?, ?B/s]

torch.Size([50263, 768])


In [ ]:
model.config

GPT2Config {
  "_attn_implementation_autoset": true,
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.0,
  "bos_token_id": 50260,
  "embd_pdrop": 0.0,
  "eos_token_id": 50257,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "pad_token_id": 50261,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.0,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "torch_dtype": "float32",
  "transformers_version": "4.51.3",
  "use_cache": true,
  "vocab_size": 50263
}

In [ ]:
model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50263, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.0, inplace=False)
          (resid_dropout): Dropout(p=0.0, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50263, bias=False)
)

In [ ]:
dataset = load_dataset('text', data_files={'train': '/content/drive/MyDrive/NLPfinal/train.txt', 'validation': '/content/drive/MyDrive/NLPfinal/valid.txt'})

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=512, padding='max_length')

In [ ]:
tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=['text'])

Map:   0%|          | 0/11225 [00:00<?, ? examples/s]

Map:   0%|          | 0/2807 [00:00<?, ? examples/s]

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
print("Dataset Info:")
print(tokenized_datasets)

Dataset Info:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 11225
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 2807
    })
})


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Model is on device: {device}")

Model is on device: cuda


In [ ]:
training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/NLPfinal/finetuned_gpt2_vietnamese',
    overwrite_output_dir=True,
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    eval_steps=2000,
    save_steps=2000,
    save_total_limit=2,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_steps=200,
    logging_dir='./logs',
    logging_steps=1000,
    load_best_model_at_end=True,
    metric_for_best_model='loss',
    greater_is_better=False,
    fp16=True,
    gradient_checkpointing=True,
    dataloader_num_workers=2,
    max_grad_norm=1.0,
    optim="adamw_torch",
    eval_strategy="steps",
    save_strategy="steps"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

In [ ]:
trainer.train()#79293fad6aed27115ec84a8a9027472dcb38da63

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nguyenquocmanh6112004 (nguyenquocmanh6112004-123) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
2000,1.487600,1.483674
4000,1.304500,1.399667
6000,1.194000,1.381048
8000,1.098500,1.379729
10000,1.042200,1.400939
12000,0.987900,1.412501
14000,0.959100,1.419344


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=14000, training_loss=1.201100581577846, metrics={'train_runtime': 7909.2086, 'train_samples_per_second': 14.192, 'train_steps_per_second': 1.775, 'total_flos': 2.9248246185984e+16, 'train_loss': 1.201100581577846, 'epoch': 9.971509971509972})

In [ ]:
model.save_pretrained('/content/drive/MyDrive/NLPfinal/finetuned_gpt2_vietnamese')
tokenizer.save_pretrained('/content/drive/MyDrive/NLPfinal/finetuned_gpt2_vietnamese')

('/content/drive/MyDrive/NLPfinal/finetuned_gpt2_vietnamese/tokenizer_config.json',
 '/content/drive/MyDrive/NLPfinal/finetuned_gpt2_vietnamese/special_tokens_map.json',
 '/content/drive/MyDrive/NLPfinal/finetuned_gpt2_vietnamese/vocab.json',
 '/content/drive/MyDrive/NLPfinal/finetuned_gpt2_vietnamese/merges.txt',
 '/content/drive/MyDrive/NLPfinal/finetuned_gpt2_vietnamese/added_tokens.json')

In [ ]:
save_path = "./final_model"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)


In [ ]:
import shutil

shutil.make_archive("final_model", 'zip', save_path)


In [ ]:
from google.colab import files
files.download("final_model.zip")


*Đánh giá mô hình qua chỉ số BLEU và ROUGE*

In [ ]:
import json
from transformers import AutoTokenizer, AutoModelForCausalLM
from nltk.translate.bleu_score import sentence_bleu
import torch

# Bước 1: Load mô hình và tokenizer đã fine-tune
checkpoint_path = "/content/drive/MyDrive/NLPfinal/finetuned_gpt2_vietnamese/checkpoint-14000"
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
model = AutoModelForCausalLM.from_pretrained(checkpoint_path)
model.eval().to("cuda" if torch.cuda.is_available() else "cpu")

# Bước 2: Đọc file valid.json
file_path = "/content/drive/MyDrive/NLPfinal/valid.json"
with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)
# Bước 3: Chuẩn bị dữ liệu
questions = [item['question'] for item in data][:10]
test_labels = [item['response'] for item in data][:10]

# Bước 4: Sinh câu trả lời từ mô hình
generated_responses = []
for question in questions:
    inputs = tokenizer.encode(question, return_tensors='pt').to(model.device)
    outputs = model.generate(inputs, max_length=150, num_beams=5, early_stopping=True)
    output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_responses.append(output_text)

# Bước 5: Tính BLEU score
bleu_scores = []
for ref, gen in zip(test_labels, generated_responses):
    bleu = sentence_bleu([ref.split()], gen.split())
    bleu_scores.append(bleu)

# Bước 6: In kết quả
average_bleu = sum(bleu_scores) / len(bleu_scores) if bleu_scores else 0
print("BLEU trung bình:", average_bleu)


BLEU trung bình: 0.07779640201040199


/usr/local/lib/python3.11/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.11/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


In [ ]:
!pip install evaluate

In [ ]:
!pip install rouge_score

In [9]:
import json
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import evaluate

# Load mô hình và tokenizer
checkpoint_path = "/content/drive/MyDrive/NLPfinal/finetuned_gpt2_vietnamese/checkpoint-14000"
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
model = AutoModelForCausalLM.from_pretrained(checkpoint_path)
model.eval().to("cuda" if torch.cuda.is_available() else "cpu")

# Load dữ liệu từ JSON
file_path = "/content/drive/MyDrive/NLPfinal/valid.json"
with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

questions = [item['question'] for item in data][:100]
references = [item['response'] for item in data][:100]
# Sinh câu trả lời từ mô hình
generated = []
for question in questions:
    inputs = tokenizer.encode(question, return_tensors='pt').to(model.device)
    outputs = model.generate(inputs, max_length=150, num_beams=5, early_stopping=True)
    output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated.append(output_text)

# Tính ROUGE
rouge = evaluate.load("rouge")
results = rouge.compute(predictions=generated, references=references)

# In kết quả ROUGE
print("Kết quả ROUGE:")
for key, value in results.items():
    print(f"{key}: {value:.4f}")


Kết quả ROUGE:
rouge1: 0.6212
rouge2: 0.3055
rougeL: 0.3444
rougeLsum: 0.3529
